# SentinelML Tree-Based Models

Compare Logistic Regression, Random Forest, default XGBoost, and Optuna-tuned XGBoost on the validation split.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.train_models import compare_all_models, compute_curve_data, load_processed_splits

In [ ]:
X_train, X_test, X_test, y_train, y_test, y_test = load_processed_splits(
    PROJECT_ROOT / "data" / "processed"
)

In [ ]:
comparison, results = compare_all_models(X_train, y_train, X_test, y_test, X_test, y_test)
comparison

## Validation ROC Curves

In [ ]:
curve_data = {
    model_name: compute_curve_data(result["model"], X_test, y_test)
    for model_name, result in results.items()
}

fig, ax = plt.subplots(figsize=(8, 6))
for model_name in comparison.index:
    data = curve_data[model_name]
    ax.plot(data["roc"]["fpr"], data["roc"]["tpr"], label=f"{model_name} ROC-AUC={comparison.loc[model_name, 'roc_auc']:.4f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Test ROC Curve")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## Validation Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for model_name in comparison.index:
    data = curve_data[model_name]
    ax.plot(
        data["precision_recall"]["recall"],
        data["precision_recall"]["precision"],
        label=f"{model_name} PR-AUC={comparison.loc[model_name, 'pr_auc']:.4f}",
    )
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Test Precision-Recall Curve")
ax.legend()
ax.grid(alpha=0.25)
plt.show()